# Stress Suite For All Methods

This notebook stress-tests all implemented methods across several situations:

- ACI with IPCW
- ACI without IPCW
- AdaFTRL
- AdaFTRL-V2 / `adaftrl_v2`
- `XuLPB`, a pure model-based online Cox LPB baseline

The suite includes ordinary stochastic censoring regimes, stronger censoring regimes, stronger model bias, and a long censor-only tail where from a switch round onward only censored observations are presented. `XuLPB` keeps `tau = alpha` fixed and uses only its online Cox survival curve, without ACI, AdaFTRL, IPCW calibration, or adaptive tau updates.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from online_survival.experiments import (
    SimulationConfig,
    behavior_svg,
    format_markdown_table,
    run_censor_only_stress_experiment,
    run_one_experiment,
    run_repeated_censor_only_stress_experiments,
    run_repeated_experiments,
    summarize_one_run,
    summarize_repeats,
)


def show_markdown(markdown):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown))
    except Exception:
        print(markdown)


def show_svg(svg):
    try:
        from IPython.display import SVG, display
        display(SVG(svg))
    except Exception:
        print(svg[:1000])


SUMMARY_COLUMNS = [
    "situation",
    "algorithm",
    "realized_coverage_mean",
    "coverage_shortfall_mean",
    "coverage_abs_error_mean",
    "average_conditional_coverage_mean",
    "average_surrogate_error_mean",
    "average_lower_bound_mean",
    "ambiguous_below_bound_fraction_mean",
    "final_tau_mean",
    "censoring_fraction_mean",
]

ONE_RUN_COLUMNS = [
    "algorithm",
    "realized_coverage",
    "target_coverage",
    "coverage_shortfall",
    "coverage_abs_error",
    "average_conditional_coverage",
    "average_surrogate_error",
    "average_lower_bound",
    "ambiguous_below_bound_fraction",
    "final_tau",
    "censoring_fraction",
]


def add_situation(rows, situation):
    return [{"situation": situation, **row} for row in rows]


def repeated_summary_for_scenario(scenario):
    if scenario["kind"] == "regular":
        repeated = run_repeated_experiments(
            scenario["config"],
            n_repeats=scenario["n_repeats"],
            seed=scenario["seed"],
            include_adaftrl_v2=True,
            include_xu_lpb=True,
        )
    else:
        repeated = run_repeated_censor_only_stress_experiments(
            scenario["config"],
            n_repeats=scenario["n_repeats"],
            seed=scenario["seed"],
            censor_only_start=scenario["censor_only_start"],
            forced_censor_fraction=scenario["forced_censor_fraction"],
            include_adaftrl_v2=True,
            include_xu_lpb=True,
        )
    return add_situation(summarize_repeats(repeated), scenario["name"])


def one_run_for_scenario(scenario):
    if scenario["kind"] == "regular":
        return run_one_experiment(
            scenario["config"],
            seed=scenario["one_run_seed"],
            include_adaftrl_v2=True,
            include_xu_lpb=True,
        )
    return run_censor_only_stress_experiment(
        scenario["config"],
        seed=scenario["one_run_seed"],
        censor_only_start=scenario["censor_only_start"],
        forced_censor_fraction=scenario["forced_censor_fraction"],
        include_adaftrl_v2=True,
        include_xu_lpb=True,
    )

## Scenarios

All scenarios use target lower coverage `1 - alpha = 0.90`. The repeated-run table below reports means across seeds. Smaller `coverage_shortfall_mean` and `coverage_abs_error_mean` are better. `average_lower_bound_mean` is the average LPB value returned by each method.

In [ ]:
scenarios = [
    {
        "name": "moderate censoring",
        "kind": "regular",
        "config": SimulationConfig(n_rounds=1000, censoring_rate=0.18, eta=0.010),
        "n_repeats": 40,
        "seed": 1100,
        "one_run_seed": 11,
    },
    {
        "name": "heavy censoring",
        "kind": "regular",
        "config": SimulationConfig(n_rounds=1000, censoring_rate=0.60, eta=0.008),
        "n_repeats": 40,
        "seed": 2100,
        "one_run_seed": 17,
    },
    {
        "name": "extreme censoring",
        "kind": "regular",
        "config": SimulationConfig(n_rounds=1000, censoring_rate=0.75, eta=0.008),
        "n_repeats": 40,
        "seed": 3100,
        "one_run_seed": 23,
    },
    {
        "name": "strong upward model bias",
        "kind": "regular",
        "config": SimulationConfig(
            n_rounds=1000,
            censoring_rate=0.45,
            eta=0.008,
            model_log_shift=0.55,
        ),
        "n_repeats": 40,
        "seed": 4100,
        "one_run_seed": 29,
    },
    {
        "name": "long censor-only tail",
        "kind": "censor_only",
        "config": SimulationConfig(n_rounds=5000, censoring_rate=0.18, eta=0.010),
        "censor_only_start": 300,
        "forced_censor_fraction": 0.20,
        "n_repeats": 20,
        "seed": 5100,
        "one_run_seed": 5,
    },
]

all_summary = []
for scenario in scenarios:
    all_summary.extend(repeated_summary_for_scenario(scenario))

show_markdown(format_markdown_table(all_summary, SUMMARY_COLUMNS, digits=4))

## Representative One-Run Figures

Each figure has numeric ticks and explicit axes. For the long censor-only tail, the vertical dashed line marks the point where the stream switches to only censored observations.

In [ ]:
for scenario in scenarios:
    records = one_run_for_scenario(scenario)
    show_markdown(f"### {scenario['name']}")
    show_markdown(format_markdown_table(summarize_one_run(records), ONE_RUN_COLUMNS, digits=4))
    show_svg(
        behavior_svg(
            records,
            window=150 if scenario["kind"] == "censor_only" else 75,
            phase_change_round=scenario.get("censor_only_start"),
            title=f"{scenario['name']}: rolling coverage and played tau",
        )
    )